In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver


from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, RemoveMessage

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI()

In [4]:
class ChatState(MessagesState):
    
    summary: str

In [5]:
def chat_node(state: ChatState):
    
    messages = []
    
    if state["summary"]:
        messages.append({
            "role": "system",
            "content": f"Conversation summary: \n{state["summary"]}"
        })
        
    messages.extend(state["messages"])
    
    print(messages)
    
    response = model.invoke(messages)
    return {"messages": [response]}

In [6]:
# Here we are making the summary and first we are deleting the extra messages that are beyond our limit
def summarize_conversation(state: ChatState):
    
    existing_summary = state["summary"]
    
    # Build summarization prompt
    if existing_summary:
        prompt = (
            f"Existing summary:\n{existing_summary}\n\n"
            "Extend the summary using the new conversation above."
        )
    else:
        prompt = "Summarize the conversation above."
        
    messages_for_summary = state["messages"] + [HumanMessage(content=prompt)]
    
    response = model.invoke(messages_for_summary)
    
    # Keep only last 2 messages verbatim
    messages_to_delete = state["messages"][:-2]
    
    return {
        "summary": response.context,
        "messages": [RemoveMessage(id=m.id) for m in messages_to_delete],
    }